# GLV Candidate Benchmark Analysis

Compares the experimental candidate branches for one GLV against their **baseline**, deciding
wins on the **medium wall-clock latency median** — the primary signal for this latency-first run.

This run is **medium-only** and **latency-first**. The medium cell is a deserialization-dominated
query (`g.V().repeat(both()).times(12)` — a large object count, ~seconds/req cross-region, so network
RTT is a negligible fraction of the wall clock). Each arm runs the medium cell across 3 sweeps and the
harness appends one ledger row per sweep. The representative latency for an arm is the **median of its
per-sweep `median` values** (median-of-medians).

**Inputs** are pulled from S3 (`s3://kirill-tp-benchmarks/cand-results/<glv>/`), where the EC2
client published `~/cand-results` after the sweep. Each arm has its own `ledger.csv`.

**The decision:** a candidate is **interesting** if its medium latency beats the baseline
(improvement > `LATENCY_IMPROVE_MIN`); the §4 table also flags whether that improvement **reproduces**
across the per-sweep medians. The bar chart in §4 is the final artifact: baseline grey, green = beats
baseline.

Restart kernel → Run All is fully reproducible: every input comes from S3 + the parameters cell.

## 0. Parameters

Edit these to retarget another GLV or run.

In [ ]:
GLV               = 'python'
BUCKET            = 'kirill-tp-benchmarks'
PREFIX            = f'cand-results/{GLV}'        # S3 prefix the EC2 client synced to
BASELINE          = 'bench-baseline'             # baseline arm = its branch-tag directory

# --- arm selection -----------------------------------------------------------
# The S3 prefix (and the local mirror) can hold stale dirs from earlier,
# incomparable runs (an old baseline + obsolete candidate sets, including other
# GLVs) that the role cannot delete. Only arms whose dir name is BASELINE or
# starts with one of ARM_INCLUDE_PREFIXES are pulled/analyzed, so this run stays
# apples-to-apples. Set this to the current GLV's candidate prefix.
ARM_INCLUDE_PREFIXES = ('cand-python-',)         # current funnel naming: auto/cand-python-<id>

# --- PRIMARY signal: medium wall-clock latency -------------------------------
# A candidate is "interesting" when its median latency beats baseline. 0.0 == any
# improvement counts; a *robust* win should also reproduce across the per-sweep
# medians (see the §4 table). Raise this to demand a minimum effect size.
LATENCY_IMPROVE_MIN = 0.0                         # min (base-cand)/base improvement to flag

SWEEPS            = [1, 2, 3]
LOCAL_DIR         = 'cand-results-data'           # where S3 objects are mirrored locally

def arm_included(arm):
    """An arm dir is in-scope if it is the baseline or matches an include-prefix."""
    return arm == BASELINE or arm.startswith(ARM_INCLUDE_PREFIXES)

## 1. Setup

In [ ]:
!pip install pandas plotly boto3 nbformat -q

import os
import boto3
import pandas as pd
import plotly.graph_objects as go

PASS_COLOR, FAIL_COLOR, BASE_COLOR = '#00CC96', '#BBBBBB', '#636EFA'
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

## 2. Download from S3

Mirrors every in-scope arm's `ledger.csv` under the prefix into a local tree `LOCAL_DIR/<arm>/...`.

The local mirror is **wiped first** so stale arms from an earlier run (e.g. a previous GLV left in
`LOCAL_DIR` on a reused Jupyter instance) can never leak into the analysis — only what is freshly
downloaded for the current `PREFIX` + `ARM_INCLUDE_PREFIXES` is considered.

In [ ]:
import shutil

def download_prefix(bucket, prefix, local_dir):
    # Wipe the local mirror first: a reused Jupyter instance may still hold arm
    # dirs from a previous run/GLV, and the arm glob below reads from disk — stale
    # dirs would otherwise be analyzed even though nothing was downloaded for them.
    shutil.rmtree(local_dir, ignore_errors=True)
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    n = 0
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            key = obj['Key']
            rel = key[len(prefix):].lstrip('/')
            arm = rel.split('/')[0]                       # first path segment = arm dir
            if not arm_included(arm):                      # skip stale/incomparable arms
                continue
            if os.path.basename(key) != 'ledger.csv':      # only the ledger is needed
                continue
            dest = os.path.join(local_dir, rel)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            s3.download_file(bucket, key, dest)
            n += 1
    return n

count = download_prefix(BUCKET, PREFIX, LOCAL_DIR)
print(f'Downloaded {count} file(s) from s3://{BUCKET}/{PREFIX} -> {LOCAL_DIR}/')
arms = sorted(d for d in os.listdir(LOCAL_DIR)
              if os.path.isdir(os.path.join(LOCAL_DIR, d))
              and not d.startswith('_') and arm_included(d))
print(f'Arms in scope ({len(arms)}):', arms)

assert BASELINE in arms, (
    f'baseline {BASELINE!r} not among arms found in {LOCAL_DIR}/: {arms}. '
    f'Set BASELINE in the parameters cell to one of these directory tags.')

## 3. Load ledgers

One tidy DataFrame, one row per cell, with an `arm` column. We keep only `candidate-eval` rows.

In [ ]:
def load_ledgers(local_dir, arms):
    frames = []
    for arm in arms:
        p = os.path.join(local_dir, arm, 'ledger.csv')
        if not os.path.exists(p):
            print(f'WARN: no ledger for {arm}'); continue
        d = pd.read_csv(p)
        d['arm'] = arm
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    if 'label' in df.columns:
        df = df[df['label'] == 'candidate-eval'].copy()
    df['is_baseline'] = df['arm'] == BASELINE
    return df

ledger = load_ledgers(LOCAL_DIR, arms)
print(f'{len(ledger)} candidate-eval rows across {ledger["arm"].nunique()} arms')
print('point_value values:', sorted(ledger['point_value'].dropna().unique().tolist()))
ledger[['arm','point_value','status','errors','median','git_sha','host']].sort_values(['arm','point_value'])

## 4. PRIMARY signal — medium wall-clock latency (the decision)

This is the latency-first decision. From the ledger we take the `point_value=='medium'` rows
(the harness appended one per sweep, so each arm has ~`len(SWEEPS)` rows). Each row's `median` is
sec/req for that sweep (clean wall-clock — these cells were **unprofiled**).

For each arm we compute the **median of its per-sweep medians** (median-of-medians) as the
representative latency, plus the per-sweep medians for a reproducibility check. A candidate's
improvement vs baseline is `(base_med - cand_med) / base_med`. A candidate is flagged **interesting**
if `improvement > LATENCY_IMPROVE_MIN`, and is **reproducible** if it also beats baseline's per-sweep
median in *every* matched sweep.

In [ ]:
# Medium rows only (label already filtered to candidate-eval in load_ledgers).
med = ledger[ledger['point_value'] == 'medium'].copy()
med['median'] = pd.to_numeric(med['median'], errors='coerce')
assert not med.empty, "no point_value=='medium' rows found in the ledger"

# Representative latency per arm = median of its per-sweep medians (median-of-medians).
lat_med = med.groupby('arm')['median'].median()
# Per-arm sorted list of per-sweep medians, for the reproducibility check.
per_sweep = med.groupby('arm')['median'].apply(lambda s: sorted(s.dropna().tolist()))

base_med = lat_med.get(BASELINE)
base_sweeps = per_sweep.get(BASELINE, [])
assert base_med is not None and pd.notna(base_med), f'no medium latency for baseline {BASELINE!r}'

def reproducible(cand_sweeps):
    """True if the candidate beats baseline's per-sweep medians in every matched sweep
    (compared rank-by-rank on the sorted per-sweep medians)."""
    if not cand_sweeps or not base_sweeps:
        return False
    k = min(len(cand_sweeps), len(base_sweeps))
    return all(cand_sweeps[i] < base_sweeps[i] for i in range(k))

cand_arms = [a for a in arms if a != BASELINE]
lat_rows = []
for a in cand_arms:
    cm = lat_med.get(a)
    improve = (base_med - cm) / base_med if pd.notna(cm) else None
    lat_rows.append({
        'candidate': a,
        'cand_med (s/req)': cm,
        'base_med (s/req)': base_med,
        'improvement': improve,
        'interesting': (improve is not None) and (improve > LATENCY_IMPROVE_MIN),
        'reproducible': reproducible(per_sweep.get(a, [])),
        'n_sweeps': len(per_sweep.get(a, [])),
    })
latency = pd.DataFrame(lat_rows).sort_values('improvement', ascending=False).reset_index(drop=True)

print(f'{GLV}: medium latency vs baseline {BASELINE!r}  (base median-of-medians = {base_med:.4f} s/req)')
print('INTERESTING:', latency.loc[latency['interesting'], 'candidate'].tolist() or 'none')

_show = latency.copy()
_show['improvement'] = _show['improvement'].map(lambda v: f'{v*100:+.2f}%' if pd.notna(v) else '—')
_show

In [ ]:
# Medium latency per arm (median-of-medians), baseline highlighted. Lower = better.
order = [BASELINE] + latency['candidate'].tolist()
_interesting = latency.set_index('candidate')['interesting']
fig = go.Figure()
fig.add_bar(
    x=[a.replace('auto/','') for a in order],
    y=[lat_med.get(a) for a in order],
    marker_color=[BASE_COLOR if a == BASELINE else
                  (PASS_COLOR if bool(_interesting.get(a, False)) else FAIL_COLOR)
                  for a in order],
    text=[f'{lat_med.get(a):.3f}' if pd.notna(lat_med.get(a)) else '' for a in order],
    textposition='outside',
)
fig.add_hline(y=base_med, line_dash='dash', line_color=BASE_COLOR,
              annotation_text=f'baseline {base_med:.3f} s/req', annotation_position='top left')
fig.update_layout(
    title=f'{GLV}: medium latency per arm (median of {len(SWEEPS)} sweeps, lower = faster; '
          f'baseline grey, green = beats baseline)',
    yaxis_title='median sec/req', height=460, showlegend=False)
fig.show()